# Repo Concierge — ask questions about this codebase

> **Note:** This agent uses a snapshot of the public `main` branch (not your local
> uncommitted changes or `data/` cache). Like any LLM, it can be wrong — verify
> important details against the repo or ask a facilitator.

**Not sure how something works? Start here.**

The repo concierge helps you **find your way** — it answers questions, points you
to the right notebooks and modules, and can quote short snippets so you know
where to dig deeper. Example questions:

- *How do I create a new data service?*
- *How do I customize the way context is presented to an LLMP?*
- *What's the difference between `backtest()` and `evaluate()`?*

It searches a committed **catalog** of the codebase (`search_repo_catalog` →
`fetch_repo_artifact`): full `aieng/forecasting`, reference implementations, and
notebooks (markdown + code cells). Domain `99_starter_agent.ipynb` notebooks are
for building forecasters; this one is your map of the repo.

Live cells are gated by `RUN_AGENT` so `Run All` is safe and free; set it to `True`
to call the model.

In [7]:
import warnings
from pathlib import Path

from IPython.display import Markdown, display  # noqa: A004


warnings.filterwarnings("ignore")

from dotenv import load_dotenv


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the workspace root."""
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "aieng-forecasting").is_dir():
            return cand
    return Path.cwd().resolve().parents[1]


ROOT = find_repo_root()
load_dotenv(ROOT / ".env", override=False)

# ── Model selection ───────────────────────────────────
# Concierge uses the lite/default model only.
AGENT_MODEL = "gemini-3.1-flash-lite-preview"

# ── Run guard ──────────────────────────────────────
RUN_AGENT = True

from getting_started.concierge_agent import build_concierge_config


print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL)

RUN_AGENT = True | model = gemini-3.1-flash-lite-preview


---
## 1. Meet the concierge

The agent uses a **catalog + artifacts** knowledge pack shipped under `concierge_agent/context/` — no build step for participants.

1. **`search_repo_catalog`** — search metadata (paths, summaries, domains); cheap, run first.
2. **`fetch_repo_artifact`** — fetch full content for a catalog path (Python modules, READMEs, notebooks with **markdown + code cells**).

Maintainers regenerate the pack from public `main` with `scripts/build_concierge_context.py` when library code or notebooks change. The `repo-navigation` skill has reference guides (no scripts).

In [8]:
from getting_started.concierge_agent import build_concierge_config


config = build_concierge_config(model=AGENT_MODEL)

print("Agent:", config.name)
print("Search enabled:    ", config.context_retrieval.enabled)
print("Code-exec enabled: ", config.code_execution.enabled)
print("Skills loaded:     ", [p.name for p in config.skills_dirs])
print("Extra tools:       ", [getattr(t, "__name__", repr(t)) for t in config.extra_tools])
display(Markdown("### System instruction\n\n*Edit in `concierge_agent/agent.py`*"))
display(Markdown(config.instruction))

Agent: repo_concierge
Search enabled:     False
Code-exec enabled:  False
Skills loaded:      ['repo-navigation']
Extra tools:        ['search_repo_catalog', 'fetch_repo_artifact']


### System instruction

*Edit in `concierge_agent/agent.py`*

## Role

You are the **repo concierge** for the agentic-forecasting bootcamp — a friendly guide who helps participants understand the repository and find their way to the right notebooks, modules, and patterns.

Answer questions clearly. Point people to **concrete paths** in the codebase (READMEs, notebooks, specs, library modules) where they can read more or try things themselves. When it helps, quote short snippets from fetched artifacts — especially from notebooks and reference implementations.

## How you work

- Ground answers in the committed catalog: call ``search_repo_catalog`` first, then ``fetch_repo_artifact`` for the paths you need (usually one to three per question).
- Prefer showing *where* something lives and *how it fits together* over long generic explanations.
- If someone is debugging or extending code, walk them through the relevant files and patterns you find in the catalog; suggest what to open next in their editor.
- Your knowledge reflects the committed public ``main`` snapshot — not the participant's local ``.env``, ``data/`` cache, or uncommitted changes. If the catalog does not cover something, say so and name the best file to open or a facilitator to ask.

## Tone

- Concise, welcoming, and practical — short paragraphs and bullet lists.
- Always cite paths returned by the catalog.


## Skills

You have one read-only skill: `repo-navigation` with reference files (catalog guide,
domain map). Load them via `load_skill_resource` when you need routing hints.

**To use a skill:**
1. Call `list_skills` → `load_skill` → `load_skill_resource` as needed.

These skills have NO scripts. Do not call `run_skill_script`.

## Repo catalog tools (required workflow)

1. **`search_repo_catalog(query, domain=None, kind=None)`** — search metadata only
   (paths, summaries, section titles). Use `domain` filters like `core.data`,
   `core.methods`, `impl.energy_oil_forecasting`, `scripts`, `docs`.
   Use `kind` filters: `python`, `notebook`, `markdown`, `yaml`.
2. **`fetch_repo_artifact(path, section=None)`** — fetch full content for one catalog
   path (optionally one heading/section). Fetch 1–3 artifacts per question.

Do not answer implementation or API questions without fetching the relevant paths.

---
## 2. Try a seed question

Edit `QUESTION` below, or jump to the next section for a multi-turn conversation.

In [9]:
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig


QUESTION = "How do I create a new data service?"

if RUN_AGENT:
    chat_agent = build_adk_agent(config)
    runner = AdkTextRunner(chat_agent, config=AdkTextRunnerConfig(app_name="repo_concierge_chat"))
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True in the setup cell to ask the concierge.")

Creating a new `DataService` in this framework typically involves registering series using an adapter and metadata. The `DataService` is responsible for fetching data, enforcing information cutoffs, and serving context to your models.

### Recommended Path
The best way to learn is to examine the provided implementations. I recommend looking at these for practical examples:

*   **Implementation Example:** `implementations/sp500_forecasting/data.py` — This shows a standard, leak-safe setup.
*   **Getting Started Guide:** `implementations/getting_started/01_cpi_data_exploration.ipynb` — The section "1. Build the DataService" walks you through the code interactively.

### The Basic Pattern
As defined in `aieng-forecasting/aieng/forecasting/data/service.py`, you follow these three steps:

1.  **Initialize the service:**
    ```python
    from aieng.forecasting.data import DataService, SeriesMetadata
    svc = DataService()
    ```
2.  **Define your adapter and metadata:**
    Choose an appropriate adapter (e.g., `StatCanAdapter` or a custom one) and instantiate `SeriesMetadata` to describe the source, frequency, and units.
3.  **Register:**
    Call `svc.register(series_id, adapter, metadata)`.

Once registered, you use `svc.context(as_of)` to produce a `ForecastContext` for your backtests, ensuring that your models cannot "look into the future" beyond the specified `as_of` date.

Do you have a specific data source in mind, or are you trying to implement a new custom adapter?

Root node repo_concierge was cancelled.


In [10]:
QUESTION = "How do I customize the way context is presented to an LLMP?"

if RUN_AGENT:
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, F821, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True to run this cell.")

To customize how context is presented to an LLM in this repository, you should look at the **`report_sources`** and **`report_ingestion`** parameters defined in the `LLMPredictorConfig` base class.

These settings are used by all `llm_processes` predictors (such as `QuantileGridLLMPredictor` or `BinaryProbabilityLLMPredictor`) to control what data is injected into the prompt.

### Key Configuration Fields
You can find these in: `aieng-forecasting/aieng/forecasting/methods/llm_processes/base.py`

*   **`report_sources`**: A `list[str]` of document source keys. When provided, the system automatically fetches documents from the `DataService` and injects them as a "CiK-style" (Context-in-Knowledge) preamble to your prompt.
*   **`report_ingestion`**:
    *   `'text'` (default): Converts the document content to markdown-formatted text before injection.
    *   `'native'`: Intended for models that support native document uploads (e.g., via multimodality).
*   **`report_max_chars`**: If you are hitting context window limits, use this to set a character truncation limit per report.

### How to use this
When you instantiate a predictor (e.g., in a notebook or experiment spec), pass these configurations into the predictor's config object:

```python
# Example of how you might pass this in a forecasting workflow
config = QuantileGridLLMPredictorConfig(
    model="gemini-3.1-flash-lite-preview",
    report_sources=["my_custom_report_key"],
    report_max_chars=20000,
    report_ingestion="text"
)
```

**Recommendation:**
If you need to change the *structure* of how that preamble is formatted (beyond just toggling ingestion modes), you should examine how the base class processes the `report_sources` list. If you find you need more advanced control (like custom system prompts or RAG-style chunking), start by exploring `aieng-forecasting/aieng/forecasting/methods/llm_processes/base.py` and see how the `Predictor` interacts with the `ForecastContext`.

Root node repo_concierge was cancelled.


In [11]:
QUESTION = "What's the difference between backtest() and evaluate()?"

if RUN_AGENT:
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, F821, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True to run this cell.")

In the agentic forecasting bootcamp, `backtest()` and `evaluate()` serve two different purposes in your development cycle. 

The core difference lies in their intent and usage constraints—specifically regarding data isolation and run budgets.

### 1. `backtest()` (Development & Tuning)
*   **Purpose:** Use this for iterative model development, feature engineering, and hyperparameter tuning.
*   **Data:** Operates on your training/development window.
*   **Constraints:** None. You can run backtests as often as you like to experiment and refine your forecasting strategy.
*   **Location:** Defined by `BacktestSpec` (e.g., `aieng-forecasting/aieng/forecasting/evaluation/backtest.py`).

### 2. `evaluate()` (Final Validation)
*   **Purpose:** Use this to estimate how well your model generalizes to **unseen, held-out data**. This is your "final exam."
*   **Data:** Operates on a protected, recent evaluation window that is meant to be untouched during the model-building phase.
*   **Constraints:** Often includes a **run budget** (`max_runs`). Using an `EvalTracker` with `evaluate()` will track your runs and prevent you from accidentally "overfitting" to the test set by running it too many times.
*   **Location:** Defined by `EvalSpec` (e.g., `aieng-forecasting/aieng/forecasting/evaluation/eval.py`).

---

### Quick Summary
| Feature | `backtest()` | `evaluate()` |
| :--- | :--- | :--- |
| **Stage** | Iterative Development | Final Generalization Check |
| **Data Window** | Training/Dev Period | Protected/Held-out Period |
| **Run Budget** | Unlimited | Often Capped (tracked via `EvalTracker`) |
| **Goal** | Improve model performance | Validate model performance |

**Pro-tip:** You can find reference `BacktestSpec` files in the `implementations/<use-case>/specs/` directories. Always look for companion `EvalSpec` files in the same location to understand what period is considered "held-out" for your specific project.

Root node repo_concierge was cancelled.


In [12]:
QUESTION = "Where should I go after getting_started if I want to build agents?"

if RUN_AGENT:
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, F821, PLE1142
    display(Markdown(reply))
else:
    print("RUN_AGENT is False — set it to True to run this cell.")

Welcome to the next steps of your agentic-forecasting journey!

Since you've completed the "getting started" phase, the best way to start building your own agents is to move from theory to our **starter agent templates**.

### Where to go
The codebase provides "hackable" starter agents that handle the boilerplate—like tool definitions, environment wiring, and proxy configuration—so you can focus on the forecasting logic.

I recommend starting with one of these implementation directories:

*   **`implementations/boc_rate_decisions/starter_agent/agent.py`**: A clean template for financial decision forecasting.
*   **`implementations/energy_oil_forecasting/starter_agent/agent.py`**: A template focused on energy commodity forecasting.
*   **`implementations/sp500_forecasting/starter_agent/agent.py`**: A template focused on market index forecasting.

### How it works
To understand the underlying "factory" that powers these agents, take a look at:
*   **`aieng-forecasting/aieng/forecasting/methods/agentic/agent_factory.py`**

This module contains the `build_adk_agent` factory function. It manages:
*   **Tool Wiring:** Attaching code execution environments and context-retrieval (web search) tools.
*   **Model Proxying:** Managing interactions with the LiteLLM/Vector proxy for Gemini models.
*   **Output Schemas:** Ensuring the agent returns the structured JSON your forecasting pipeline expects (see `aieng/forecasting/methods/agentic/outputs.py` for those schemas).

**Pro-tip:** When you're ready to build, copy one of the `starter_agent/` directories into your own workspace and start modifying the system prompts and tool configurations in `agent.py`. If you run into installation errors, remember that agentic functionality requires the `agentic` extra: 

```bash
pip install aieng-forecasting[agentic]
```

Which domain or forecasting problem are you most interested in exploring? I can point you toward the specific experiment tasks or data schemas associated with that path.

Root node repo_concierge was cancelled.


---
## 3. Terminal mode — multi-turn conversations

For extended back-and-forth, use the ADK CLI in the integrated terminal. From this
directory (`implementations/getting_started/`):

```bash
cd implementations/getting_started
uv run adk run concierge_agent
```

That loads the same `repo_concierge` agent (`gemini-3.1-flash-lite-preview`) with
`search_repo_catalog`, `fetch_repo_artifact`, and the repo-navigation skill.

**Alternative:** `uv run adk web concierge_agent` opens a browser UI (same agent).

---

**Where next?** Forecasting starter agents live in each domain implementation's
`99_starter_agent.ipynb` (food, energy, BoC, S&P 500). This concierge only explains
the repo — open one of those when you're ready to build and score a forecaster.